# Feature Engineering Pipeline

This notebook performs end-to-end feature engineering on the cleaned mental health dataset (`mental_health_cleaned.csv`).

### Pipeline Overview:
1. **Drop Excluded Columns**: Remove high-cardinality, population-specific (`University`, `Department`) and aggregate score columns (`Stress Value`, `Anxiety Value`, `Depression Value`).
2. **Impute Gender**: Handle `'Prefer not to say'` using mode imputation.
3. **Collapse Age Fringe Groups**: Merge `'Below 18'` into `'18-22'` and `'Above 30'` into `'27-30'`.
4. **Encode Demographics**: Apply One-Hot Encoding (`Gender`, `Age`, `waiver_or_scholarship`) with `drop_first=True` and Ordinal Encoding (`Academic_Year`, `Current_CGPA`) as `int64` with median assignment (value `2`) for `'Other'`.
5. **Verify Item Dtypes**: Cast item response columns (PSS1–10, GAD1–7, PHQ1–9) from float to integer (`int64`).
6. **Define Feature Matrix & Targets**: Construct feature matrix $X$ and separate target vectors $y_{stress}$, $y_{anxiety}$, $y_{depression}$.
7. **Save Outputs**: Export $X$, individual target series, and the full engineered dataset to CSV files.
8. **Pipeline Summary & Verification**: Display final matrix shapes, column dtypes, preview of ordinal features, and target class distributions.

## Setup & Data Loading

Load required libraries (`pandas`, `numpy`, `os`) and load the cleaned dataset `mental_health_cleaned.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np

# Path to cleaned dataset
data_path = "mental_health_cleaned.csv"

if not os.path.exists(data_path):
    print(f"Error: '{data_path}' not found.")
else:
    df = pd.read_csv(data_path)
    print(f"Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

## Step 1 — Drop Excluded Columns

**Rationale:**
- `University` & `Department`: Dropped due to high cardinality and heavy population skew (CS/IUB/AIUB overrepresentation), which would harm model generalizability.
- `Stress Value`, `Anxiety Value`, `Depression Value`: Dropped because aggregate sum scores are directly redundant when individual scale items (PSS1–10, GAD1–7, PHQ1–9) are included as features.

**Action:** Retain all 26 item scores, demographic variables, and target label columns.

In [ ]:
exclude_cols = ['University', 'Department', 'Stress Value', 'Anxiety Value', 'Depression Value']
df_feat = df.drop(columns=exclude_cols)

print(f"Original columns: {df.shape[1]}")
print(f"Remaining columns after dropping excluded set: {df_feat.shape[1]}")
print("\nRemaining columns:")
print(list(df_feat.columns))

## Step 2 — Impute Gender 'Prefer not to say'

**Rationale:**
The `'Prefer not to say'` category contains only 10 rows (~0.5% of dataset), which is too small to form a statistically viable separate category for modeling. Mode imputation replaces these rows with the majority class (`Male`).

**Action:** Impute mode gender and verify value distributions.

In [ ]:
# Calculate mode among valid gender responses
gender_mode = df_feat[df_feat['Gender'] != 'Prefer not to say']['Gender'].mode()[0]
print(f"Mode Gender value for imputation: '{gender_mode}'")

# Replace 'Prefer not to say' with mode
df_feat['Gender'] = df_feat['Gender'].replace({'Prefer not to say': gender_mode})

print("\nGender value counts after mode imputation:")
print(df_feat['Gender'].value_counts())

## Step 3 — Handle Age Fringe Groups

**Rationale:**
- `'Below 18'` (4 rows) and `'Above 30'` (3 rows) are tiny fringe categories in an otherwise 18–26 dominated university student dataset (~96.4%).
- Keeping them as separate bins creates sparse features with high variance.

**Action:** Collapse `'Below 18'` into the adjacent `'18-22'` bin, and `'Above 30'` into the adjacent `'27-30'` bin.

In [ ]:
age_collapse_map = {
    'Below 18': '18-22',
    'Above 30': '27-30'
}

df_feat['Age'] = df_feat['Age'].replace(age_collapse_map)

print("Age group value counts after collapsing fringe categories:")
print(df_feat['Age'].value_counts())

## Step 4 — Encode Demographics & Fix Ordinal Encoding

**Rationale:**
- **One-Hot Encoding**: Applied to nominal categorical variables (`Gender`, `Age`, `waiver_or_scholarship`) with `drop_first=True` to prevent multi-collinearity / dummy variable trap.
- **Ordinal Encoding**: Applied to naturally ordered categories (`Academic_Year`, `Current_CGPA`).
  - `Academic_Year`: `'First Year or Equivalent'` (0) < `'Second Year or Equivalent'` (1) < `'Third Year or Equivalent'` (2) < `'Fourth Year or Equivalent'` (3).
  - `Current_CGPA`: `'Below 2.50'` (0) < `'2.50 - 2.99'` (1) < `'3.00 - 3.39'` (2) < `'3.40 - 3.79'` (3) < `'3.80 - 4.00'` (4).
  - `'Other'` categories: Assigned the **median ordinal value 2** (the scale midpoint across 5 categories 0..4) and cast explicitly to `int64` integer type.

**Action:** Execute ordinal mapping as `int64` and perform one-hot encoding for nominal categories.

In [ ]:
# 1. Ordinal Encoding with 'Other' = 2 (median of 5-category scale 0..4) cast to int64
academic_year_map = {
    'First Year or Equivalent': 0,
    'Second Year or Equivalent': 1,
    'Third Year or Equivalent': 2,
    'Fourth Year or Equivalent': 3,
    'Other': 2  # Median ordinal value for 5-category scale
}

cgpa_map = {
    'Below 2.50': 0,
    '2.50 - 2.99': 1,
    '3.00 - 3.39': 2,
    '3.40 - 3.79': 3,
    '3.80 - 4.00': 4,
    'Other': 2  # Median ordinal value for 5-category scale
}

df_feat['Academic_Year'] = df_feat['Academic_Year'].map(academic_year_map).astype('int64')
df_feat['Current_CGPA'] = df_feat['Current_CGPA'].map(cgpa_map).astype('int64')

print("=== Academic_Year Fixed Value Counts (int64) ===")
print(df_feat['Academic_Year'].value_counts())
print(f"Academic_Year Dtype: {df_feat['Academic_Year'].dtype}")

print("\n=== Current_CGPA Fixed Value Counts (int64) ===")
print(df_feat['Current_CGPA'].value_counts())
print(f"Current_CGPA Dtype: {df_feat['Current_CGPA'].dtype}")

# 2. One-Hot Encoding for Gender, Age, waiver_or_scholarship
ohe_target_cols = ['Gender', 'Age', 'waiver_or_scholarship']
df_encoded = pd.get_dummies(df_feat, columns=ohe_target_cols, drop_first=True, dtype=int)

print(f"\nDataframe shape after One-Hot Encoding: {df_encoded.shape}")

## Step 5 — Verify Item Score Dtypes

**Rationale:**
Individual item scores (PSS1–10, GAD1–7, PHQ1–9) are discrete integer responses (0–4 for PSS, 0–3 for GAD and PHQ). Converting float columns to integer improves memory efficiency and data cleanliness.

**Action:** Verify existing dtypes and cast all 26 item score columns to integer (`int64`).

In [ ]:
item_cols = [f'PSS{i}' for i in range(1, 11)] + [f'GAD{i}' for i in range(1, 8)] + [f'PHQ{i}' for i in range(1, 10)]

print("Item score dtypes before casting:")
print(df_encoded[item_cols].dtypes.value_counts())

# Cast to int64
df_encoded[item_cols] = df_encoded[item_cols].astype('int64')

print("\nItem score dtypes after casting:")
print(df_encoded[item_cols].dtypes.value_counts())

## Step 6 — Define Feature Matrix X and Target Variables

**Rationale:**
Separate the predictive feature matrix $X$ (demographics + item scores) from the target label series ($y_{stress}$, $y_{anxiety}$, $y_{depression}$) to prepare for model training.

**Action:** Extract $X$ and individual target vectors, confirming matrix shapes and class distributions.

In [ ]:
target_names = ['Stress Label', 'Anxiety Label', 'Depression Label']

# Separate target series
y_stress = df_encoded['Stress Label']
y_anxiety = df_encoded['Anxiety Label']
y_depression = df_encoded['Depression Label']

# Feature matrix X
X = df_encoded.drop(columns=target_names)

print(f"Feature matrix X shape: {X.shape[0]} rows, {X.shape[1]} features")
print("\nTarget y_stress value counts:")
print(y_stress.value_counts())
print("\nTarget y_anxiety value counts:")
print(y_anxiety.value_counts())
print("\nTarget y_depression value counts:")
print(y_depression.value_counts())

## Step 7 — Resave All Output Files

**Rationale:**
Overwrite previously saved CSV files with the corrected, fully engineered integer data assets.

**Action:** Re-export `X_features.csv`, target files (`y_stress.csv`, `y_anxiety.csv`, `y_depression.csv`), and rebuild `mental_health_engineered.csv` ($X$ + all three targets appended).

In [ ]:
# Rebuild full engineered dataframe: X + all three target columns appended
df_engineered_rebuilt = pd.concat([X, y_stress, y_anxiety, y_depression], axis=1)

# Export corrected CSVs
X.to_csv('X_features.csv', index=False)
y_stress.to_csv('y_stress.csv', index=False)
y_anxiety.to_csv('y_anxiety.csv', index=False)
y_depression.to_csv('y_depression.csv', index=False)
df_engineered_rebuilt.to_csv('mental_health_engineered.csv', index=False)

print(f"[OK] X_features.csv successfully saved: {X.shape[0]} rows, {X.shape[1]} columns")
print(f"[OK] y_stress.csv successfully saved: {y_stress.shape[0]} rows, 1 column")
print(f"[OK] y_anxiety.csv successfully saved: {y_anxiety.shape[0]} rows, 1 column")
print(f"[OK] y_depression.csv successfully saved: {y_depression.shape[0]} rows, 1 column")
print(f"[OK] mental_health_engineered.csv successfully saved: {df_engineered_rebuilt.shape[0]} rows, {df_engineered_rebuilt.shape[1]} columns")

## Step 8 — Final Verification & Summary

Verification of $X$ shape, dtypes for all features, preview of fixed ordinal columns, and target class distributions.

In [ ]:
print("=" * 65)
print("          FINAL FEATURE MATRIX VERIFICATION")
print("=" * 65)
print(f"X.shape: {X.shape[0]} rows, {X.shape[1]} columns\n")

print("=== X.dtypes for all columns ===")
print(X.dtypes.to_string())

print("\n=== First 5 rows of X[['Academic_Year', 'Current_CGPA']] ===")
print(X[['Academic_Year', 'Current_CGPA']].head())

print("\n" + "-" * 65)
print("CONFIRMATION OF SAVED OUTPUT FILES:")
print("-" * 65)
files_to_check = ['X_features.csv', 'y_stress.csv', 'y_anxiety.csv', 'y_depression.csv', 'mental_health_engineered.csv']
for fname in files_to_check:
    if os.path.exists(fname):
        temp_df = pd.read_csv(fname)
        print(f"  [OK] {fname:<30}: {temp_df.shape[0]} rows, {temp_df.shape[1]} columns")
    else:
        print(f"  [X]  {fname:<30}: NOT FOUND")
print("=" * 65)